# Tube MPC Demo — Robust Model Predictive Control

**Companion notebook for Chapter 10** (Robust MPC & Tube MPC) of the MPC lecture notes by İ. Küçükdemiral.

This notebook demonstrates the complete Tube MPC design pipeline:
1. LQR feedback gain computation
2. Minimal robust positively invariant (mRPI) set computation
3. Constraint tightening (Pontryagin difference)
4. Terminal set computation (maximal invariant set)
5. Tube MPC formulation and closed-loop simulation with disturbances

**System:** Double integrator $x^+ = Ax + Bu + w$, with $x = [\text{position}, \text{velocity}]^\top$, $u = \text{acceleration}$, $w \in \mathcal{W}$ bounded.

📖 [Lecture notes](https://kucukdemiral.github.io/model-predictive-control.html) · 📥 [tube_mpc_demo.py](https://kucukdemiral.github.io/tube_mpc_demo.py)

In [ ]:
# Install dependencies (required for Google Colab)
!pip install numpy matplotlib control cvxpy piqp \
    --find-links https://github.com/PREDICT-EPFL/MPC-Course-EPFL/releases/expanded_assets/mpt4py_wheels \
    --find-links https://github.com/PREDICT-EPFL/MPC-Course-EPFL/releases/expanded_assets/pycddlib_wheels \
    mpt4py==0.1.5

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpt4py import Polyhedron
from control import dlqr
import cvxpy as cp

%config InlineBackend.figure_format = 'retina'

## Step 0: System Definition

Double integrator: $x^+ = Ax + Bu + w$

$$
A = \begin{bmatrix} 1 & 1 \\ 0 & 1 \end{bmatrix}, \quad
B = \begin{bmatrix} 1 \\ 0.5 \end{bmatrix}
$$

with constraints $|x_i| \le 3$, $|u| \le 0.5$, and disturbance $|w_i| \le 0.1$.

In [ ]:
A = np.array([[1, 1], [0, 1]])
B = np.array([[1], [0.5]])
Q = np.eye(2)
R = 10 * np.eye(1)

nx, nu = B.shape

x_max = 3.0
u_max = 0.5
w_max = 0.1

print(f'System: {nx} states, {nu} inputs')
print(f'Constraints: |x_i| ≤ {x_max}, |u| ≤ {u_max}, |w_i| ≤ {w_max}')

## Step 1: LQR Feedback Gain

The tube controller uses $u_k = v_k^\star + K(x_k - z_k^\star)$, where $K$ is the LQR gain that keeps the error $e_k = x_k - z_k$ small.

In [ ]:
K, Qf, _ = dlqr(A, B, Q, R)
K = -K  # convention: u = Kx (negative feedback)
A_cl = A + B @ K

eigs = np.linalg.eigvals(A_cl)
print(f'LQR gain K = {K.flatten()}')
print(f'Closed-loop eigenvalues: {eigs}')
print(f'All stable: {all(abs(e) < 1 for e in eigs)}')

## Step 2: Constraint Sets as Polytopes

We represent constraints as H-polytopes: $\mathcal{X} = \{x \mid Hx \le h\}$.

In [ ]:
X = Polyhedron.from_Hrep(A=np.vstack((np.eye(nx), -np.eye(nx))),
                         b=x_max * np.ones(2 * nx))
U = Polyhedron.from_Hrep(A=np.vstack((np.eye(nu), -np.eye(nu))),
                         b=u_max * np.ones(2 * nu))
W = Polyhedron.from_Hrep(A=np.vstack((np.eye(nx), -np.eye(nx))),
                         b=w_max * np.ones(2 * nx))

print(f'X: {X.A.shape[0]} half-planes')
print(f'U: {U.A.shape[0]} half-planes')
print(f'W: {W.A.shape[0]} half-planes')

## Step 3: Minimal Robust Positively Invariant (mRPI) Set

The mRPI set is the smallest set $\mathcal{E}$ such that $A_{cl}\mathcal{E} \oplus \mathcal{W} \subseteq \mathcal{E}$. It captures the worst-case disturbance accumulation:

$$
\mathcal{E}_\infty = \bigoplus_{i=0}^{\infty} A_{cl}^i \mathcal{W} = \mathcal{W} \oplus A_{cl}\mathcal{W} \oplus A_{cl}^2\mathcal{W} \oplus \cdots
$$

We iterate until $\|A_{cl}^i\|_2$ is negligible.

In [ ]:
def min_robust_invariant_set(A_cl, W, max_iter=30, tol=1e-2):
    """Compute the mRPI set via iterative Minkowski sums."""
    Omega = W
    for i in range(1, max_iter + 1):
        A_cl_i = np.linalg.matrix_power(A_cl, i)
        Omega_next = Omega + A_cl_i @ W  # Minkowski sum
        Omega_next.minHrep()

        if np.linalg.norm(A_cl_i, ord=2) < tol:
            print(f'mRPI converged after {i} iterations '
                  f'(||A_cl^{i}||₂ = {np.linalg.norm(A_cl_i, ord=2):.2e})')
            return Omega_next

        Omega = Omega_next

    print(f'WARNING: mRPI did NOT converge after {max_iter} iterations')
    return Omega

E = min_robust_invariant_set(A_cl, W)
print(f'E has {E.A.shape[0]} half-planes')

In [ ]:
# Visualise X and E
fig, ax = plt.subplots(figsize=(6, 5))
X.plot(ax, color='green', opacity=0.3, label=r'$\mathcal{X}$')
E.plot(ax, color='red', opacity=0.5, label=r'$\mathcal{E}$ (mRPI)')
ax.set_xlabel(r'$x_1$'); ax.set_ylabel(r'$x_2$')
ax.set_title('State constraints and mRPI set')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_aspect('equal')
plt.show()

## Step 4: Constraint Tightening

Tighten the constraints by the mRPI set to leave room for the tube:

$$
\tilde{\mathcal{X}} = \mathcal{X} \ominus \mathcal{E}, \qquad
\tilde{\mathcal{U}} = \mathcal{U} \ominus K\mathcal{E}
$$

The nominal MPC plans within these tighter sets. The tube around the nominal trajectory then guarantees the real state stays in $\mathcal{X}$.

In [ ]:
# Tightened state constraints
X_tilde = X - E  # Pontryagin difference

# Tightened input constraints
KE = E.affine_map(K)  # image of E under K

# Method 1: direct Pontryagin difference
U_tilde_1 = U - KE

# Method 2: manual tightening via support function
U_tilde_b = U.b.copy()
for i in range(U_tilde_b.shape[0]):
    U_tilde_b[i] -= KE.support(U.A[i, :])
U_tilde_2 = Polyhedron.from_Hrep(A=U.A, b=U_tilde_b)

print(f'Methods agree: {U_tilde_1 == U_tilde_2}')
U_tilde = U_tilde_1

print(f'X̃: {X_tilde.A.shape[0]} half-planes, non-empty: {not X_tilde.is_empty}')
print(f'Ũ: {U_tilde.A.shape[0]} half-planes, non-empty: {not U_tilde.is_empty}')

In [ ]:
# Visualise constraint tightening
fig, ax = plt.subplots(figsize=(6, 5))
X.plot(ax, color='green', opacity=0.2, label=r'$\mathcal{X}$')
X_tilde.plot(ax, color='gold', opacity=0.4,
             label=r'$\tilde{\mathcal{X}} = \mathcal{X} \ominus \mathcal{E}$')
E.plot(ax, color='red', opacity=0.5, label=r'$\mathcal{E}$')
ax.set_xlabel(r'$x_1$'); ax.set_ylabel(r'$x_2$')
ax.set_title('Constraint tightening')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_aspect('equal')
plt.show()

## Step 5: Terminal Sets

Compute the maximal invariant set $\mathcal{X}_f$ (for nominal MPC) and $\tilde{\mathcal{X}}_f$ (for tube MPC) inside the respective tightened constraints. The terminal set ensures recursive feasibility.

In [ ]:
def max_invariant_set(A_cl, X, max_iter=30):
    """Compute the maximal invariant set for x+ = A_cl x inside X."""
    O = X
    for i in range(1, max_iter + 1):
        F, f = O.A, O.b
        O_next = Polyhedron.from_Hrep(
            np.vstack((F, F @ A_cl)),
            np.concatenate((f, f))
        )
        O_next.minHrep()
        if O_next == O:
            print(f'Converged after {i} iterations')
            return O_next
        print(f'Iteration {i}... not yet converged')
        O = O_next
    print(f'WARNING: did NOT converge after {max_iter} iterations')
    return O

# Nominal terminal set
print('=== Nominal terminal set ===')
X_and_KU = X.intersect(Polyhedron.from_Hrep(U.A @ K, U.b))
Xf = max_invariant_set(A_cl, X_and_KU)

# Tube terminal set
print('\n=== Tube terminal set ===')
X_tilde_and_KU_tilde = X_tilde.intersect(
    Polyhedron.from_Hrep(U_tilde.A @ K, U_tilde.b)
)
Xf_tilde = max_invariant_set(A_cl, X_tilde_and_KU_tilde)

In [ ]:
# Visualise terminal sets
fig, ax = plt.subplots(figsize=(6, 5))
Xf.plot(ax, color='green', opacity=0.3, label=r'$\mathcal{X}_f$ (nominal)')
Xf_tilde.plot(ax, color='steelblue', opacity=0.5,
              label=r'$\tilde{\mathcal{X}}_f$ (tube)')
ax.set_xlabel(r'$x_1$'); ax.set_ylabel(r'$x_2$')
ax.set_title('Terminal sets: nominal vs tube')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_aspect('equal')
plt.show()

## Step 6: Tube MPC Formulation

The nominal MPC optimises over $z_k$ (nominal states) and $v_k$ (nominal inputs) subject to the tightened constraints. The initial condition $z_0$ is constrained so that $x_0 \in z_0 \oplus \mathcal{E}$.

$$
\min_{z, v} \sum_{k=0}^{N-1} \bigl[ z_k^\top Q z_k + v_k^\top R v_k \bigr] + z_N^\top Q_f z_N
$$
subject to:
- $z_{k+1} = A z_k + B v_k$
- $z_k \in \tilde{\mathcal{X}}$, $v_k \in \tilde{\mathcal{U}}$
- $z_N \in \tilde{\mathcal{X}}_f$
- $x_0 \in z_0 \oplus \mathcal{E}$

In [ ]:
N = 10  # prediction horizon

z_var = cp.Variable((N + 1, nx), name='z')
v_var = cp.Variable((N, nu), name='v')
x0_var = cp.Parameter((nx,), name='x0')

# Cost
cost = 0
for k in range(N):
    cost += cp.quad_form(z_var[k], Q)
    cost += cp.quad_form(v_var[k], R)
cost += cp.quad_form(z_var[-1], Qf)

# Constraints
constraints = []
constraints.append(E.A @ (x0_var - z_var[0]) <= E.b)       # x₀ ∈ z₀ ⊕ E
constraints.append(z_var[1:].T == A @ z_var[:-1].T + B @ v_var.T)  # dynamics
constraints.append(X_tilde.A @ z_var[:-1].T <= X_tilde.b.reshape(-1, 1))  # states
constraints.append(U_tilde.A @ v_var.T <= U_tilde.b.reshape(-1, 1))  # inputs
constraints.append(Xf_tilde.A @ z_var[-1] <= Xf_tilde.b)  # terminal

tube_mpc = cp.Problem(cp.Minimize(cost), constraints)
print(f'Tube MPC problem formulated (N = {N})')

## Step 7: Closed-Loop Simulation

Run 20 Monte Carlo realisations with random disturbances $w_k \sim \text{Uniform}(\mathcal{W})$.

At each step, the tube control law is:
$$
u_k = v_0^\star + K(x_k - z_0^\star)
$$

In [ ]:
num_samples = 20
N_sim = 40

x0 = Xf_tilde.sample(1).flatten()
print(f'Initial state: x₀ = [{x0[0]:.3f}, {x0[1]:.3f}]')

x_trajs = np.zeros((num_samples, N_sim + 1, nx))
u_trajs = np.zeros((num_samples, N_sim, nu))

for i in range(num_samples):
    x_trajs[i, 0] = x0
    xk = x0.copy()

    for k in range(N_sim):
        x0_var.value = xk
        tube_mpc.solve(cp.PIQP)
        assert tube_mpc.status == cp.OPTIMAL, \
            f'Solver returned: {tube_mpc.status} at sample {i}, step {k}'

        z0 = z_var[0].value
        v0 = v_var[0].value
        uk = v0 + K @ (xk - z0)     # tube law
        wk = W.sample(1).flatten()    # random disturbance
        xk = A @ xk + B @ uk + wk   # true dynamics

        x_trajs[i, k + 1] = xk.flatten()
        u_trajs[i, k] = uk.flatten()

print('Simulation complete.')
print(f'State violations: {np.any(np.abs(x_trajs) > x_max + 1e-6)}')
print(f'Input violations: {np.any(np.abs(u_trajs) > u_max + 1e-6)}')

In [ ]:
# Time trajectories
t = np.arange(N_sim + 1)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

for x_traj, u_traj in zip(x_trajs, u_trajs):
    axes[0].plot(t, x_traj[:, 0], linewidth=0.7, alpha=0.6)
    axes[1].plot(t, x_traj[:, 1], linewidth=0.7, alpha=0.6)
    axes[2].plot(t[:-1], u_traj[:, 0], linewidth=0.7, alpha=0.6)

for i in [0, 1]:
    axes[i].axhline(x_max, color='red', ls='--', lw=1.5, label='constraint')
    axes[i].axhline(-x_max, color='red', ls='--', lw=1.5)
axes[2].axhline(u_max, color='red', ls='--', lw=1.5, label='constraint')
axes[2].axhline(-u_max, color='red', ls='--', lw=1.5)

axes[0].set_ylabel(r'$x_1$ (position)')
axes[1].set_ylabel(r'$x_2$ (velocity)')
axes[2].set_ylabel(r'$u$ (input)')
axes[2].set_xlabel('Time step $k$')

for ax in axes:
    ax.grid(True, alpha=0.3); ax.legend(loc='upper right')

fig.suptitle(f'Closed-loop trajectories ({num_samples} realisations)',
             fontsize=14, fontweight='bold')
fig.align_ylabels(); fig.tight_layout()
plt.show()

In [ ]:
# Phase portrait
fig, ax = plt.subplots(figsize=(7, 6))
X.plot(ax, opacity=0.1, color='green', label=r'$\mathcal{X}$')
X_tilde.plot(ax, opacity=0.15, color='steelblue', label=r'$\tilde{\mathcal{X}}$')
Xf_tilde.plot(ax, opacity=0.2, color='gold', label=r'$\tilde{\mathcal{X}}_f$')

for x_traj in x_trajs:
    ax.plot(x_traj[:, 0], x_traj[:, 1], linewidth=0.7, alpha=0.5)
ax.plot(x0[0], x0[1], 'ko', markersize=6, label=r'$x_0$')

ax.set_xlabel(r'$x_1$ (position)'); ax.set_ylabel(r'$x_2$ (velocity)')
ax.set_title('Phase portrait — all trajectories stay inside $\mathcal{X}$')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3); ax.set_aspect('equal')
fig.tight_layout()
plt.show()

## Key Takeaways

- The **mRPI set** $\mathcal{E}$ captures the worst-case disturbance accumulation under the feedback gain $K$
- **Constraint tightening** ($\tilde{\mathcal{X}} = \mathcal{X} \ominus \mathcal{E}$) leaves room for the tube
- The **tube law** $u = v^\star + K(x - z^\star)$ keeps the real state inside the tube
- All constraints are satisfied despite random disturbances — **robust constraint satisfaction guaranteed**

### Exercises

Try modifying the parameters above and re-running:
1. Increase `w_max` to 0.3 — what happens to the tightened constraints?
2. Change `R` from 10 to 1 — how does a more aggressive gain affect the tube width?
3. Reduce `x_max` to 1.5 — at what point does the tightened set become empty?